# 04_transformer_block
先阅读对应讲解，预测形状后运行。修改参数，再写下观察。

In [5]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "exercises/week03/block_utils.py").exists():
    if root == root.parent:
        raise RuntimeError("请在 ai_basic 项目中启动 Notebook")
    root = root.parent
sys.path.insert(0, str(root / "exercises/week03"))


In [6]:
from block_utils import X, positions, block, show
import numpy as np

result = block(X + positions(*X.shape))
for name, value in result.items():
    show(name, value)
assert result["Y"].shape == X.shape
assert np.isfinite(result["Y"]).all()


X shape=(3, 4)
[[1.         1.         0.         2.        ]
 [0.84147098 1.54030231 0.00999983 1.99995   ]
 [1.90929743 0.58385316 0.01999867 0.99980001]]

normalized shape=(3, 4)
[[ 0.          0.         -1.41419942  1.41419942]
 [-0.34128444  0.58868689 -1.44776653  1.20036408]
 [ 1.49557141 -0.42700956 -1.24489074  0.17632889]]

attention shape=(3, 4)
[[-0.39724936 -0.70902491  0.10161837  0.45205564]
 [-0.40505328 -0.70068855  0.09180018  0.45487575]
 [-0.42683314 -0.66043462  0.08401306  0.44618849]]

weights shape=(2, 3, 3)
[[[0.34862588 0.29615986 0.35521426]
  [0.30712717 0.24002961 0.45284322]
  [0.31923558 0.64277074 0.03799368]]

 [[0.42594176 0.38047696 0.19358128]
  [0.40856377 0.36128607 0.23015016]
  [0.3719678  0.36178722 0.26624498]]]

H shape=(3, 4)
[[ 0.60275064  0.29097509  0.10161837  2.45205564]
 [ 0.43641771  0.83961375  0.10180002  2.45482575]
 [ 1.48246429 -0.07658146  0.10401173  1.4459885 ]]

ffn shape=(3, 4)
[[-0.36340507  0.38051295  0.41406127 -0.366777

## 我的解释
运行前预测：

实际观察：

原因与疑问：

## PyTorch 对照

`exercises/week03/torch_block_example.py` 是同一个 Pre-LN Block 的 PyTorch 实现。内核必须用项目 `.venv`（已装 torch 2.14），否则会报 `ModuleNotFoundError: No module named 'torch'`。

In [7]:
import sys
import torch

print(sys.executable)
print("torch:", torch.__version__)
assert "ai-agent-learning" in sys.executable, "内核不是项目 .venv，请改用 start_notebook.bat 启动"

C:\Users\chende\Documents\A-Codes\Github\ai-agent-learning\.venv\Scripts\python.exe
torch: 2.14.1+cu126


In [8]:
# 方式一：作为子进程直接运行脚本（触发 __main__ 分支）
import subprocess
script = root / "exercises/week03/torch_block_example.py"
r = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

output: torch.Size([1, 3, 4]) weights: torch.Size([1, 2, 3, 3])




In [9]:
# 方式二：导入类，逐步查看中间结果（更适合学习）
from torch_block_example import TransformerBlock

torch.manual_seed(3)
model = TransformerBlock().eval()
for name, p in model.named_parameters():
    print(f"{name:20s} {tuple(p.shape)}")

ln1.weight           (4,)
ln1.bias             (4,)
ln2.weight           (4,)
ln2.bias             (4,)
attn.in_proj_weight  (12, 4)
attn.in_proj_bias    (12,)
attn.out_proj.weight (4, 4)
attn.out_proj.bias   (4,)
ffn.0.weight         (8, 4)
ffn.0.bias           (8,)
ffn.2.weight         (4, 8)
ffn.2.bias           (4,)


In [10]:
x = torch.randn(1, 3, 4)          # (batch, tokens, d_model)
with torch.no_grad():
    z = model.ln1(x)                # (1, 3, 4)  沿特征维归一化
    a, w = model.attn(z, z, z, need_weights=True, average_attn_weights=False)
    h = x + a                       # (1, 3, 4)  残差 1，加的是原始 x
    y = h + model.ffn(model.ln2(h)) # (1, 3, 4)  残差 2
    y2, w2 = model(x)
for name, t in [("x", x), ("z=LN1(x)", z), ("attention", a), ("H", h), ("Y", y)]:
    print(f"{name:12s} {tuple(t.shape)}")
print("weights", tuple(w.shape))
print("每行权重和（应全为 1）:", w.sum(-1).flatten().tolist())
print("逐步计算与 model(x) 一致:", torch.allclose(y, y2))

x            (1, 3, 4)
z=LN1(x)     (1, 3, 4)
attention    (1, 3, 4)
H            (1, 3, 4)
Y            (1, 3, 4)
weights (1, 2, 3, 3)
每行权重和（应全为 1）: [1.0, 1.0, 0.9999999403953552, 0.9999999403953552, 1.0, 1.0]
逐步计算与 model(x) 一致: True


In [ ]:
w[0]  # 每个头的 (tokens, tokens) 注意力权重，无 causal mask

### 与 NumPy 版的差异

| 维度 | `block_utils.block` | `TransformerBlock` |
| --- | --- | --- |
| 输入形状 | `(n, d)` | `(batch, n, d)` |
| 权重形状 | `(heads, n, n)` | `(batch, heads, n, n)` |
| Wq/Wk/Wv | 每头独立生成 | 打包为 `in_proj_weight (12, 4)` |
| 投影偏置 | 无 | 有（`bias=True`） |
| LayerNorm gamma/beta | 写死 1 / 0 | 可学习 `weight` / `bias` |
| 位置编码 | 在 Block 外部加 | 在 Block 外部加 |
| 训练能力 | 无（纯前向演示） | 有（标准 `nn.Module`） |

两者结构对应但参数不同，数值不可直接比较。

自查：为什么 `need_weights=True` 会让 PyTorch 走慢一些的实现路径？`average_attn_weights=False` 关掉了什么？

## 用魔法命令执行脚本

`%run` 在**内核当前工作目录**下解析相对路径，而本 Notebook 位于 `notebooks/week03/`，因此不能写 `%run torch_block_example.py`。用第一个单元格求出的 `root` 拼绝对路径最稳妥。

`%run` 会把脚本当作 `__main__` 执行，所以 `if __name__ == "__main__":` 分支会跑；它与 `subprocess` 的区别是**共享同一个命名空间**，脚本里的 `model`、`y`、`weights` 会留在 Notebook 全局变量中，可能覆盖你自己的同名变量。

In [11]:
%run {root}/exercises/week03/torch_block_example.py

output: torch.Size([1, 3, 4]) weights: torch.Size([1, 2, 3, 3])


In [12]:
import os, sys
print("内核 cwd =", os.getcwd())
print("内核解释器 =", sys.executable)

内核 cwd = C:\Users\chende\Documents\A-Codes\Github\ai-agent-learning\notebooks\week03
内核解释器 = C:\Users\chende\Documents\A-Codes\Github\ai-agent-learning\.venv\Scripts\python.exe


In [13]:
# 排查用：三种执行方式都能跑通，但都依赖绝对路径
script = root / "exercises/week03/torch_block_example.py"

print("[1] %run    ->", end=" ")
get_ipython().run_line_magic("run", str(script))

out = subprocess.run([sys.executable, str(script)],
                    capture_output=True, text=True).stdout.strip()
print("[2] subprocess ->", out)

print("[3] 相对路径 torch_block_example.py 会失败，因为 cwd =", os.getcwd())

[1] %run    -> output: torch.Size([1, 3, 4]) weights: torch.Size([1, 2, 3, 3])
[2] subprocess -> output: torch.Size([1, 3, 4]) weights: torch.Size([1, 2, 3, 3])
[3] 相对路径 torch_block_example.py 会失败，因为 cwd = C:\Users\chende\Documents\A-Codes\Github\ai-agent-learning\notebooks\week03


In [14]:
# 脚本把 y 和 weights 留在了全局命名空间里
print("y", tuple(y.shape))
print("weights", tuple(weights.shape))
print("model", type(model).__name__)

print(y)
print(weights)

y (1, 3, 4)
weights (1, 2, 3, 3)
model TransformerBlock
tensor([[[-2.6360, -1.6663,  0.4287, -0.1142],
         [-1.6540, -0.4726,  0.1180, -0.1280],
         [-0.4831, -0.4770, -1.1965, -1.0547]]])
tensor([[[[0.3271, 0.3106, 0.3623],
          [0.3039, 0.2907, 0.4055],
          [0.3967, 0.4081, 0.1952]],

         [[0.2228, 0.2150, 0.5622],
          [0.2094, 0.2011, 0.5895],
          [0.4318, 0.4535, 0.1148]]]])


### 魔法命令小结

| 写法 | 作用 | 说明 |
| --- | --- | --- |
| `%run 路径.py` | 在当前内核里执行脚本 | 触发 `__main__`，共享全局命名空间 |
| `%run -i 路径.py` | 同上，出错不中断 | 调试脚本时用 |
| `%run -m 包.模块` | 以模块方式运行 | 支持包内相对导入 |
| `!python 路径.py` | 开子进程执行 | 隔离命名空间，但用 `python` 命令，未必是 `.venv` |
| `%load_ext autoreload` + `%autoreload 2` | 重载已导入模块 | 改了 `.py` 后免重启内核 |
| `%cd 路径` | 切换内核工作目录 | **全局生效**，重启内核才复位 |
| `import os; os.getcwd()` | 查看工作目录 | `%pwd` 存在但静默，不打印 |

改了 `torch_block_example.py` 之后，用 `%load_ext autoreload` + `%autoreload 2` 可以让 `import` 重新读取文件，不必重启内核。

### 三个常见报错

| 报错 | 原因 | 解决 |
| --- | --- | --- |
| `[WinError 3] 系统找不到指定的路径` | `%cd` 相对层数算错。`..` 只退一层；从 `notebooks/week03` 出发退 3 层会到 `A-Codes` | `%cd ..\\..\exercises\week03`，或直接用 `{root}` |
| `UsageError: Line magic function `%` not found` | `%` 后多了空格，`% python x.py` 被当成 magic 名 `python` | 去掉空格用 `%run`，或改用 `!python` |
| `UsageError: Cell magic `%%x` not found` | 用了 `%%`，那是 cell magic | 单个 `%` 是行 magic，`%%` 只用于 `%%time`、`%%bash` 等 |

记住：内核 cwd 默认是 **Notebook 所在目录**（这里是 `notebooks/week03`），不是项目根目录。所有路径都别写死相对路径，用第一个单元格算出的 `root`。